In [1]:
# Εγκατάσταση απαραίτητων βιβλιοθηκών
import pandas as pd
import numpy as np
import os
import zipfile
import pandas as pd
import os

In [2]:
!pip install aif360
!pip install aif360 BlackBoxAuditing
from aif360.datasets import StandardDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import DisparateImpactRemover

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for BlackBoxAuditing: filename=BlackBoxAuditing-0.1.54-py2.py3-none-any.whl size=1394756 sha256=8a68a88c1cdd6ed54c524a47739efee2f58b71b5f2c1a42021f5c6294f267b51
  Stored in directory: /root/.cache/pip/wheels/c9/8c/03/073e80e604151fb4cdc68b2e56a97f338d7723e4a4ab5e3823
Successfully built BlackBoxAuditing


pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


In [3]:
zip_path = "/content/reduced_datasets.zip"
extract_path = "/content/preterm_dataset_versions"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Αρχεία αποσυμπιέστηκαν στο:", extract_path)
print(os.listdir(extract_path))

folder_path = '/content/preterm_dataset_versions/content'

# Δείξε όλα τα περιεχόμενα του φακέλου
print("Αρχεία μέσα στο φάκελο:")
for f in os.listdir(folder_path):
    print(f)

Αρχεία αποσυμπιέστηκαν στο: /content/preterm_dataset_versions
['content']
Αρχεία μέσα στο φάκελο:
preterm_dataset_versions


In [5]:
# Φάκελος με τα 30 datasets
input_folder = '/content/preterm_dataset_versions/content/preterm_dataset_versions/'

# Λίστα για να αποθηκεύσουμε τα fairness αποτελέσματα
results = []

# Loop για όλα τα αρχεία
for i in range(1, 31):
    file_path = os.path.join(input_folder, f'reduced_preterm_dataset_{i}.xlsx')

    # Ανάγνωση αρχείου
    df = pd.read_excel(file_path)

    # Δημιουργία δυαδικής μεταβλητής ηλικίας
    df['Maternal age binary'] = (df['Maternal age'] >= 35).astype(int)

    # Δημιουργία AIF360 dataset
    dataset = StandardDataset(df,
                              label_name='Preterm_birth',
                              favorable_classes=[0],  # Θετικό αποτέλεσμα
                              protected_attribute_names=['Maternal age binary'],
                              privileged_classes=[[0]])  # <35 ετών ως προνομιούχες

    # Ορισμός ομάδων
    unprivileged_groups = [{'Maternal age binary': 1}]
    privileged_groups = [{'Maternal age binary': 0}]

    # Υπολογισμός fairness metrics
    metric = BinaryLabelDatasetMetric(dataset,
                                      unprivileged_groups=unprivileged_groups,
                                      privileged_groups=privileged_groups)

    # Αποθήκευση των αποτελεσμάτων
    results.append({
        'Dataset': f'reduced_preterm_dataset_{i}',
        'Disparate Impact': metric.disparate_impact(),
        'Statistical Parity Difference': metric.statistical_parity_difference()
    })

# Δημιουργία πίνακα με τα αποτελέσματα
results_df = pd.DataFrame(results)

# Προβολή
print(results_df)

# Αποθήκευση των αποτελεσμάτων σε CSV
results_df.to_csv('maternal_age_bias_metrics.csv', index=False)

# Κατέβασμα του αποτελέσματος
from google.colab import files
files.download('maternal_age_bias_metrics.csv')

                       Dataset  Disparate Impact  \
0    reduced_preterm_dataset_1          0.927454   
1    reduced_preterm_dataset_2          0.942732   
2    reduced_preterm_dataset_3          0.937603   
3    reduced_preterm_dataset_4          0.932511   
4    reduced_preterm_dataset_5          0.942732   
5    reduced_preterm_dataset_6          0.963612   
6    reduced_preterm_dataset_7          0.974277   
7    reduced_preterm_dataset_8          0.985097   
8    reduced_preterm_dataset_9          0.990566   
9   reduced_preterm_dataset_10          0.963612   
10  reduced_preterm_dataset_11          1.012846   
11  reduced_preterm_dataset_12          1.041637   
12  reduced_preterm_dataset_13          0.947897   
13  reduced_preterm_dataset_14          0.958336   
14  reduced_preterm_dataset_15          0.968925   
15  reduced_preterm_dataset_16          0.953098   
16  reduced_preterm_dataset_17          0.968925   
17  reduced_preterm_dataset_18          0.974277   
18  reduced_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Φάκελος με τα αρχεία
input_folder = '/content/preterm_dataset_versions/content/preterm_dataset_versions'

# Λίστα για αποθήκευση αποτελεσμάτων
results = []

# Βρόχος για κάθε αρχείο
for i in range(1, 31):
    file_path = os.path.join(input_folder, f'reduced_preterm_dataset_{i}.xlsx')

    try:
        df = pd.read_excel(file_path)

        # Εκτύπωση μέσων όρων για γρήγορη επιβεβαίωση
        print(f"\n--- Dataset {i} ---")
        print(df.groupby('Smoking')['Preterm_birth'].mean())

        # Dataset AIF360
        dataset = BinaryLabelDataset(
            df=df[['Smoking', 'Preterm_birth']].dropna(),
            label_names=['Preterm_birth'],
            protected_attribute_names=['Smoking']
        )

        # Ορισμός ομάδων
        privileged = [{'Smoking': 1}]
        unprivileged = [{'Smoking': 0}]

        # Υπολογισμός μετρικών
        metric = BinaryLabelDatasetMetric(dataset,
                                          privileged_groups=privileged,
                                          unprivileged_groups=unprivileged)

        disparate_impact = metric.disparate_impact()
        stat_parity_diff = metric.statistical_parity_difference()

        # Εκτύπωση
        print("Bias Metrics for Smoking:")
        print(f"Disparate Impact: {disparate_impact:.4f}")
        print(f"Statistical Parity Difference: {stat_parity_diff:.4f}")

        # Αποθήκευση αποτελέσματος
        results.append({
            'Dataset': i,
            'Disparate Impact': disparate_impact,
            'Statistical Parity Difference': stat_parity_diff
        })

    except Exception as e:
        print(f"Σφάλμα στο dataset {i}: {e}")

# Μετατροπή σε DataFrame
results_df = pd.DataFrame(results)

# Προβολή
print(results_df)

# Αποθήκευση των αποτελεσμάτων σε CSV
results_df.to_csv("/content/smoking_bias_metrics.csv", index=False)

# Κατέβασμα του αποτελέσματος
from google.colab import files
files.download('smoking_bias_metrics.csv')


--- Dataset 1 ---
Smoking
0.0    0.191724
1.0    0.226667
Name: Preterm_birth, dtype: float64
Bias Metrics for Smoking:
Disparate Impact: 0.8458
Statistical Parity Difference: -0.0349

--- Dataset 2 ---
Smoking
0.0    0.189488
1.0    0.246753
Name: Preterm_birth, dtype: float64
Bias Metrics for Smoking:
Disparate Impact: 0.7679
Statistical Parity Difference: -0.0573

--- Dataset 3 ---
Smoking
0.0    0.184979
1.0    0.283951
Name: Preterm_birth, dtype: float64
Bias Metrics for Smoking:
Disparate Impact: 0.6514
Statistical Parity Difference: -0.0990

--- Dataset 4 ---
Smoking
0.0    0.189488
1.0    0.246753
Name: Preterm_birth, dtype: float64
Bias Metrics for Smoking:
Disparate Impact: 0.7679
Statistical Parity Difference: -0.0573

--- Dataset 5 ---
Smoking
0.0    0.187240
1.0    0.265823
Name: Preterm_birth, dtype: float64
Bias Metrics for Smoking:
Disparate Impact: 0.7044
Statistical Parity Difference: -0.0786

--- Dataset 6 ---
Smoking
0.0    0.192837
1.0    0.216216
Name: Preterm_bi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Φορτώνουμε τα αποτελέσματα
smoking_df = pd.read_csv("/content/smoking_bias_metrics.csv")
maternal_df = pd.read_csv("/content/maternal_age_bias_metrics.csv")  # αν έχεις αποθηκεύσει τα metrics

# Ορισμός των ορίων
lower_bound = 0.8
upper_bound = 1.2

# Φιλτράρισμα DI μεταξύ 0.8 και 1.2
smoking_valid = smoking_df[(smoking_df['Disparate Impact'] >= lower_bound) & (smoking_df['Disparate Impact'] <= upper_bound)]
maternal_valid = maternal_df[(maternal_df['Disparate Impact'] >= lower_bound) & (maternal_df['Disparate Impact'] <= upper_bound)]

# Εμφάνιση αριθμού datasets που πληρούν το κριτήριο
print(f"Datasets με Disparate Impact μεταξύ 0.8 και 1.2 για Smoking: {len(smoking_valid)} / {len(smoking_df)}")
print(f"Datasets με Disparate Impact μεταξύ 0.8 και 1.2 για Maternal Age: {len(maternal_valid)} / {len(maternal_df)}")

Datasets με Disparate Impact μεταξύ 0.8 και 1.2 για Smoking: 8 / 30
Datasets με Disparate Impact μεταξύ 0.8 και 1.2 για Maternal Age: 30 / 30


In [8]:
di_values = []

# Φάκελος με τα αρχεία
input_folder = '/content/preterm_dataset_versions/content/preterm_dataset_versions'

for i in range(1, 31):
    try:
        file_path = os.path.join(input_folder, f'reduced_preterm_dataset_{i}.xlsx')

        df = pd.read_excel(file_path)

        # Φιλτράρουμε NaN
        temp_df = df[['Smoking', 'Preterm_birth']].dropna()

        # Δημιουργία BinaryLabelDataset
        dataset = BinaryLabelDataset(df=temp_df,
                                     label_names=['Preterm_birth'],
                                     protected_attribute_names=['Smoking'])

        # Υπολογισμός DI
        metric = BinaryLabelDatasetMetric(dataset,
                                          privileged_groups=[{'Smoking': 1}],
                                          unprivileged_groups=[{'Smoking': 0}])

        di = metric.disparate_impact()
        di_values.append(di)

        print(f"Dataset {i} - DI: {di:.4f}")

    except Exception as e:
        print(f"Σφάλμα στο dataset_{i}: {e}")

# Υπολογισμός συνολικού μέσου όρου DI
if di_values:
    overall_avg_di = np.mean(di_values)
    print(f"\nΣυνολικός Μέσος Όρος DI για Smoking: {overall_avg_di:.4f}")
else:
    print("Δεν υπολογίστηκε κανένα DI.")

Dataset 1 - DI: 0.8458
Dataset 2 - DI: 0.7679
Dataset 3 - DI: 0.6514
Dataset 4 - DI: 0.7679
Dataset 5 - DI: 0.7044
Dataset 6 - DI: 0.8919
Dataset 7 - DI: 0.6066
Dataset 8 - DI: 1.0713
Dataset 9 - DI: 1.0031
Dataset 10 - DI: 0.6768
Dataset 11 - DI: 0.5866
Dataset 12 - DI: 0.8919
Dataset 13 - DI: 0.7346
Dataset 14 - DI: 0.7044
Dataset 15 - DI: 0.7346
Dataset 16 - DI: 0.8048
Dataset 17 - DI: 0.6066
Dataset 18 - DI: 0.6281
Dataset 19 - DI: 0.7044
Dataset 20 - DI: 0.7044
Dataset 21 - DI: 0.6768
Dataset 22 - DI: 0.8458
Dataset 23 - DI: 0.8919
Dataset 24 - DI: 0.6281
Dataset 25 - DI: 0.7679
Dataset 26 - DI: 0.7044
Dataset 27 - DI: 0.7679
Dataset 28 - DI: 0.7346
Dataset 29 - DI: 0.6281
Dataset 30 - DI: 0.7679

Συνολικός Μέσος Όρος DI για Smoking: 0.7500


In [9]:
import scipy.stats as stats

# Φάκελος με τα αρχεία
input_folder = '/content/preterm_dataset_versions/content/preterm_dataset_versions'

# Λίστες για αποθήκευση των αποτελεσμάτων
fisher_results = []
chi2_results = []

for i in range(1, 31):
    try:
        # Διαδρομή προς το αρχείο
        file_path = os.path.join(input_folder, f'reduced_preterm_dataset_{i}.xlsx')
        df = pd.read_excel(file_path)

        # Αφαίρεση NaN
        temp_df = df[['Smoking', 'Preterm_birth']].dropna()

        # Δημιουργία contingency table
        contingency_table = pd.crosstab(temp_df['Smoking'], temp_df['Preterm_birth'])

        # Υπολογισμός Fisher's Exact Test (μόνο για πίνακες 2x2)
        if contingency_table.shape == (2, 2):
            odds_ratio, fisher_p = stats.fisher_exact(contingency_table)
        else:
            fisher_p = None

        # Υπολογισμός Chi-Square Test
        chi2, chi2_p, dof, expected = stats.chi2_contingency(contingency_table)

        # Αποθήκευση αποτελεσμάτων
        fisher_results.append({'Dataset': f'dataset_{i}', 'Fisher p-value': fisher_p})
        chi2_results.append({'Dataset': f'dataset_{i}', 'Chi-Square p-value': chi2_p})

    except Exception as e:
        print(f"Σφάλμα στο dataset_{i}: {e}")

# Δημιουργία DataFrames
fisher_df = pd.DataFrame(fisher_results)
chi2_df = pd.DataFrame(chi2_results)

# Εμφάνιση αποτελεσμάτων
print("Fisher's Exact Test Results:")
print(fisher_df)

print("\nChi-Square Test Results:")
print(chi2_df)

Fisher's Exact Test Results:
       Dataset  Fisher p-value
0    dataset_1        0.447137
1    dataset_2        0.228053
2    dataset_3        0.038474
3    dataset_4        0.228053
4    dataset_5        0.100600
5    dataset_6        0.644340
6    dataset_7        0.012832
7    dataset_8        0.876219
8    dataset_9        1.000000
9   dataset_10        0.072976
10  dataset_11        0.008245
11  dataset_12        0.644340
12  dataset_13        0.174635
13  dataset_14        0.100600
14  dataset_15        0.174635
15  dataset_16        0.360737
16  dataset_17        0.012832
17  dataset_18        0.026342
18  dataset_19        0.100600
19  dataset_20        0.100600
20  dataset_21        0.072976
21  dataset_22        0.447137
22  dataset_23        0.644340
23  dataset_24        0.026342
24  dataset_25        0.228053
25  dataset_26        0.100600
26  dataset_27        0.228053
27  dataset_28        0.174635
28  dataset_29        0.026342
29  dataset_30        0.228053

Chi-Squar

In [10]:
# Φόρτωση των αποτελεσμάτων
smoking_df = pd.read_csv("/content/smoking_bias_metrics.csv")

# Υπολογισμός μέσου όρου του Disparate Impact
mean_di = smoking_df["Disparate Impact"].mean()

print(f"Μέσος όρος Disparate Impact για Smoking: {mean_di:.4f}")

Μέσος όρος Disparate Impact για Smoking: 0.7500


In [11]:
import scipy.stats as stats

# Φάκελος με τα σύνολα δεδομένων
output_folder = '/content/preterm_dataset_versions/content/preterm_dataset_versions'

# Λίστες για αποθήκευση των αποτελεσμάτων
fisher_results = []
chi_square_results = []

# Επανάληψη για τα 30 σύνολα
for i in range(1, 31):
    # Ανάγνωση του κάθε dataset
    file_path = os.path.join(output_folder, f'reduced_preterm_dataset_{i}.xlsx')
    df = pd.read_excel(file_path)

    # Δημιουργία πίνακα συχνοτήτων (contingency table) για Smoking και Preterm_birth
    contingency_table = pd.crosstab(df['Smoking'], df['Preterm_birth'])

    # Fisher's Exact Test
    oddsratio, fisher_p_value = stats.fisher_exact(contingency_table)

    # Chi-Square Test
    chi2, chi_p_value, dof, expected = stats.chi2_contingency(contingency_table)

    # Αποθήκευση των αποτελεσμάτων
    fisher_results.append({'Dataset': f'dataset_{i}', 'Fisher p-value': fisher_p_value})
    chi_square_results.append({'Dataset': f'dataset_{i}', 'Chi-Square p-value': chi_p_value})

# Δημιουργία DataFrame και αποθήκευση σε CSV
fisher_df = pd.DataFrame(fisher_results)
chi_square_df = pd.DataFrame(chi_square_results)

# Αποθήκευση σε αρχεία CSV
fisher_df.to_csv('/content/fisher_exact_results.csv', index=False)
chi_square_df.to_csv('/content/chi_square_results.csv', index=False)

# Εκτύπωση των αποτελεσμάτων
print(fisher_df)
print(chi_square_df)

       Dataset  Fisher p-value
0    dataset_1        0.447137
1    dataset_2        0.228053
2    dataset_3        0.038474
3    dataset_4        0.228053
4    dataset_5        0.100600
5    dataset_6        0.644340
6    dataset_7        0.012832
7    dataset_8        0.876219
8    dataset_9        1.000000
9   dataset_10        0.072976
10  dataset_11        0.008245
11  dataset_12        0.644340
12  dataset_13        0.174635
13  dataset_14        0.100600
14  dataset_15        0.174635
15  dataset_16        0.360737
16  dataset_17        0.012832
17  dataset_18        0.026342
18  dataset_19        0.100600
19  dataset_20        0.100600
20  dataset_21        0.072976
21  dataset_22        0.447137
22  dataset_23        0.644340
23  dataset_24        0.026342
24  dataset_25        0.228053
25  dataset_26        0.100600
26  dataset_27        0.228053
27  dataset_28        0.174635
28  dataset_29        0.026342
29  dataset_30        0.228053
       Dataset  Chi-Square p-value
0   

In [12]:
# Δημιουργία του DataFrame με τα αποτελέσματα του Fisher's Exact Test και του Chi-Square Test
fisher_p_values = [
    0.447137, 0.228053, 0.038474, 0.228053, 0.100600, 0.644340, 0.012832, 0.876219,
    1.000000, 0.072976, 0.008245, 0.644340, 0.174635, 0.100600, 0.174635, 0.360737,
    0.012832, 0.026342, 0.100600, 0.100600, 0.072976, 0.447137, 0.644340, 0.026342,
    0.228053, 0.100600, 0.228053, 0.174635, 0.026342, 0.228053
]

chi_square_p_values = [
    0.565950, 0.291686, 0.047317, 0.291686, 0.127502, 0.741735, 0.014962, 0.913792,
    1.000000, 0.079264, 0.007936, 0.741735, 0.196864, 0.127502, 0.196864, 0.414717,
    0.014962, 0.027138, 0.127502, 0.127502, 0.079264, 0.565950, 0.741735, 0.027138,
    0.291686, 0.127502, 0.291686, 0.196864, 0.027138, 0.291686
]

# Δημιουργία DataFrame
df = pd.DataFrame({
    'Dataset': [f'dataset_{i+1}' for i in range(30)],
    'Fisher p-value': fisher_p_values,
    'Chi-Square p-value': chi_square_p_values
})

# Φιλτράρισμα των datasets με p-value < 0.05
fisher_significant = df[df['Fisher p-value'] < 0.05]
chi_square_significant = df[df['Chi-Square p-value'] < 0.05]

# Εκτύπωση των αποτελεσμάτων
print(f"Fisher's Exact Test - Συνολικός αριθμός datasets με στατιστική συσχέτιση: {len(fisher_significant)}/30")
print(f"Chi-Square Test - Συνολικός αριθμός datasets με στατιστική συσχέτιση: {len(chi_square_significant)}/30")

# Επιπλέον, αν θέλεις να δεις τα datasets με στατιστική συσχέτιση
print("\nDatasets με στατιστική συσχέτιση (Fisher's Exact Test):")
print(fisher_significant[['Dataset', 'Fisher p-value']])

print("\nDatasets με στατιστική συσχέτιση (Chi-Square Test):")
print(chi_square_significant[['Dataset', 'Chi-Square p-value']])

Fisher's Exact Test - Συνολικός αριθμός datasets με στατιστική συσχέτιση: 7/30
Chi-Square Test - Συνολικός αριθμός datasets με στατιστική συσχέτιση: 7/30

Datasets με στατιστική συσχέτιση (Fisher's Exact Test):
       Dataset  Fisher p-value
2    dataset_3        0.038474
6    dataset_7        0.012832
10  dataset_11        0.008245
16  dataset_17        0.012832
17  dataset_18        0.026342
23  dataset_24        0.026342
28  dataset_29        0.026342

Datasets με στατιστική συσχέτιση (Chi-Square Test):
       Dataset  Chi-Square p-value
2    dataset_3            0.047317
6    dataset_7            0.014962
10  dataset_11            0.007936
16  dataset_17            0.014962
17  dataset_18            0.027138
23  dataset_24            0.027138
28  dataset_29            0.027138


In [ ]:
# Φάκελος με τα datasets
data_folder = "/content/preterm_dataset_versions/content/preterm_dataset_versions"

# Λίστες για αποτελέσματα
results = []

# Επανάληψη για κάθε dataset
for i in range(1, 31):
    file_path = os.path.join(data_folder, f"reduced_preterm_dataset_{i}.xlsx")

    try:
        # Ανάγνωση dataset
        df = pd.read_excel(file_path)

        # Αφαίρεση NaN
        df = df[['History_of_PTD', 'Preterm_birth']].dropna()

        # Dataset για AIF360
        dataset = StandardDataset(df,
                                  label_name='Preterm_birth',
                                  favorable_classes=[0],
                                  protected_attribute_names=['History_of_PTD'],
                                  privileged_classes=[[0]])

        # Ορισμός groups
        unprivileged_groups = [{'History_of_PTD': 1}]
        privileged_groups = [{'History_of_PTD': 0}]

        # Υπολογισμός metrics
        metric = BinaryLabelDatasetMetric(dataset,
                                          unprivileged_groups=unprivileged_groups,
                                          privileged_groups=privileged_groups)

        # Αποθήκευση αποτελεσμάτων
        results.append({
            'Dataset': f'dataset_{i}',
            'Disparate Impact': metric.disparate_impact(),
            'Statistical Parity Difference': metric.statistical_parity_difference()
        })

    except Exception as e:
        print(f"Σφάλμα με το αρχείο dataset_{i}: {e}")

# Δημιουργία dataframe αποτελεσμάτων
results_df = pd.DataFrame(results)

# Προβολή
print(results_df)

# Αποθήκευση των αποτελεσμάτων σε CSV
results_df.to_csv("/content/history_of_PTD_bias_metrics.csv", index=False)

# Κατέβασμα του αποτελέσματος
from google.colab import files
files.download('history_of_PTD_bias_metrics.csv')

       Dataset  Disparate Impact  Statistical Parity Difference
0    dataset_1               0.0                      -0.819338
1    dataset_2               0.0                      -0.817259
2    dataset_3               0.0                      -0.817259
3    dataset_4               0.0                      -0.816223
4    dataset_5               0.0                      -0.817259
5    dataset_6               0.0                      -0.815190
6    dataset_7               0.0                      -0.811083
7    dataset_8               0.0                      -0.816223
8    dataset_9               0.0                      -0.815190
9   dataset_10               0.0                      -0.821429
10  dataset_11               0.0                      -0.808030
11  dataset_12               0.0                      -0.818297
12  dataset_13               0.0                      -0.820382
13  dataset_14               0.0                      -0.816223
14  dataset_15               0.0        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Φάκελος όπου βρίσκονται τα αρχεία
data_folder = "/content/preterm_dataset_versions/content/preterm_dataset_versions"

results = []

for i in range(1, 31):
    file_path = os.path.join(data_folder, f"reduced_preterm_dataset_{i}.xlsx")

    try:
        df = pd.read_excel(file_path)

        # Δημιουργία δυαδικής μεταβλητής Short_Cervix
        df = df[['Cervical length', 'Preterm_birth']].dropna()
        df['Short_Cervix'] = (df['Cervical length'] < 15).astype(int)

        # Δημιουργία AIF360 dataset
        dataset = StandardDataset(
            df,
            label_name='Preterm_birth',
            favorable_classes=[0],
            protected_attribute_names=['Short_Cervix'],
            privileged_classes=[[0]]
        )

        # Υπολογισμός fairness metrics
        metric = BinaryLabelDatasetMetric(
            dataset,
            privileged_groups=[{'Short_Cervix': 0}],
            unprivileged_groups=[{'Short_Cervix': 1}]
        )

        results.append({
            'Dataset': f'dataset_{i}',
            'Disparate Impact': metric.disparate_impact(),
            'Statistical Parity Difference': metric.statistical_parity_difference()
        })

    except Exception as e:
        print(f"Σφάλμα με το αρχείο dataset_{i}: {e}")

# Δημιουργία dataframe αποτελεσμάτων
results_df = pd.DataFrame(results)

# Προβολή
print(results_df)

# Αποθήκευση των αποτελεσμάτων σε CSV
results_df.to_csv("/content/cervical_length_bias_metrics.csv", index=False)

# Κατέβασμα του αποτελέσματος
from google.colab import files
files.download('cervical_length_bias_metrics.csv')

       Dataset  Disparate Impact  Statistical Parity Difference
0    dataset_1               0.0                      -0.854111
1    dataset_2               0.0                      -0.858667
2    dataset_3               0.0                      -0.850727
3    dataset_4               0.0                      -0.849604
4    dataset_5               0.0                      -0.840731
5    dataset_6               0.0                      -0.847368
6    dataset_7               0.0                      -0.858667
7    dataset_8               0.0                      -0.855246
8    dataset_9               0.0                      -0.849604
9   dataset_10               0.0                      -0.852980
10  dataset_11               0.0                      -0.845144
11  dataset_12               0.0                      -0.850727
12  dataset_13               0.0                      -0.847368
13  dataset_14               0.0                      -0.850727
14  dataset_15               0.0        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Φάκελος με τα datasets
data_folder = "/content/preterm_dataset_versions/content/preterm_dataset_versions"

# Λίστα για τα αποτελέσματα
results = []

for i in range(1, 31):
    file_path = os.path.join(data_folder, f"reduced_preterm_dataset_{i}.xlsx")

    try:
        df = pd.read_excel(file_path)

        # Επιλογή σχετικών στηλών και αφαίρεση NaNs
        df_temp = df[['Placental_cord_insertion', 'Preterm_birth']].dropna()

        # Ορισμός privileged/unprivileged ομάδων
        privileged_groups = [{'Placental_cord_insertion': 0}]
        unprivileged_groups = [{'Placental_cord_insertion': 3}]

        # Μετατροπή σε BinaryLabelDataset
        dataset = BinaryLabelDataset(
            df=df_temp,
            label_names=['Preterm_birth'],
            protected_attribute_names=['Placental_cord_insertion']
        )

        # Υπολογισμός μετρικών
        metric = BinaryLabelDatasetMetric(
            dataset,
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups
        )

        results.append({
            'Dataset': f'dataset_{i}',
            'Disparate Impact': metric.disparate_impact(),
            'Statistical Parity Difference': metric.statistical_parity_difference()
        })

    except Exception as e:
        print(f"Σφάλμα με το αρχείο dataset_{i}: {e}")

# Δημιουργία dataframe αποτελεσμάτων
results_df = pd.DataFrame(results)

# Προβολή
print(results_df)

# Αποθήκευση των αποτελεσμάτων σε CSV
results_df.to_csv("/content/placental_cord_insertion_bias_metrics.csv", index=False)

# Κατέβασμα του αποτελέσματος
from google.colab import files
files.download('placental_cord_insertion_bias_metrics.csv')

/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=privileged)
/usr/local/lib/python3.11/dist-packages/aif360/metrics/binary_label_dataset_metric.py:105: RuntimeWarning: invalid value encountered in scalar divide
  return (self.num_positives(privileged=pr

       Dataset  Disparate Impact  Statistical Parity Difference
0    dataset_1         38.333333                       0.973913
1    dataset_2         29.000000                       0.965517
2    dataset_3         19.666667                       0.949153
3    dataset_4               NaN                            NaN
4    dataset_5         23.400000                       0.957265
5    dataset_6               NaN                            NaN
6    dataset_7         29.000000                       0.965517
7    dataset_8         18.230769                       0.945148
8    dataset_9         29.000000                       0.965517
9   dataset_10               NaN                            NaN
10  dataset_11         29.000000                       0.965517
11  dataset_12               NaN                            NaN
12  dataset_13         19.666667                       0.949153
13  dataset_14               NaN                            NaN
14  dataset_15         25.888889        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Φάκελος που περιέχει τα datasets
data_folder = "/content/preterm_dataset_versions/content/preterm_dataset_versions"

# Λίστα για αποθήκευση των αποτελεσμάτων
results = []

# Ορισμός privileged/unprivileged groups
privileged_groups = [{'Gravida': x} for x in range(0, 3)]     # 0, 1, 2
unprivileged_groups = [{'Gravida': x} for x in range(3, 8)]   # 3 έως 7

# Βρόχος για όλα τα datasets
for i in range(1, 31):
    file_path = os.path.join(data_folder, f"reduced_preterm_dataset_{i}.xlsx")

    try:
        df = pd.read_excel(file_path)

        # Επιλογή σχετικών στηλών και αφαίρεση NaN
        df_temp = df[['Gravida', 'Preterm_birth']].dropna()

        # Δημιουργία BinaryLabelDataset
        dataset = BinaryLabelDataset(
            df=df_temp,
            label_names=['Preterm_birth'],
            protected_attribute_names=['Gravida']
        )

        # Υπολογισμός fairness metrics
        metric = BinaryLabelDatasetMetric(
            dataset,
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups
        )

        # Αποθήκευση αποτελεσμάτων
        results.append({
            'Dataset': f'dataset_{i}',
            'Disparate Impact': metric.disparate_impact(),
            'Statistical Parity Difference': metric.statistical_parity_difference()
        })

    except Exception as e:
        print(f"Σφάλμα με το αρχείο dataset_{i}: {e}")

# Δημιουργία dataframe αποτελεσμάτων
results_df = pd.DataFrame(results)

# Προβολή
print(results_df)

# Αποθήκευση των αποτελεσμάτων σε CSV
results_df.to_csv("/content/gravida_bias_metrics.csv", index=False)

# Κατέβασμα του αποτελέσματος
from google.colab import files
files.download('gravida_bias_metrics.csv')

       Dataset  Disparate Impact  Statistical Parity Difference
0    dataset_1          1.594891                       0.110734
1    dataset_2          1.729966                       0.134258
2    dataset_3          1.455790                       0.085847
3    dataset_4          1.729966                       0.134258
4    dataset_5          1.455790                       0.085847
5    dataset_6          1.861566                       0.156541
6    dataset_7          1.594891                       0.110734
7    dataset_8          1.455790                       0.085847
8    dataset_9          1.594891                       0.110734
9   dataset_10          1.796168                       0.145548
10  dataset_11          1.729966                       0.134258
11  dataset_12          1.312057                       0.059459
12  dataset_13          2.053521                       0.187871
13  dataset_14          1.594891                       0.110734
14  dataset_15          1.594891        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>